In [0]:
dbutils.widgets.text("source_catalog", "workspace")
dbutils.widgets.text("source_schema", "default")
dbutils.widgets.text("source_volume", "retail")
dbutils.widgets.text("source_file", "online-retail-dataset.csv")

dbutils.widgets.text("target_catalog", "workspace")
dbutils.widgets.text("target_schema", "bronze")
dbutils.widgets.text("target_table", "invoices_bronze")

source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
source_volume = dbutils.widgets.get("source_volume")
source_file = dbutils.widgets.get("source_file")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")

source_path = f"/Volumes/{source_catalog}/{source_schema}/{source_volume}/{source_file}"
target_table_fqn = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
# Lectura do ficheiro incluíndo metadatos
# Crear o esquema de destino
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

# COMMAND ----------

# Definición do esquema do CSV
schema = T.StructType([
    T.StructField("InvoiceNo", T.StringType(), True),
    T.StructField("StockCode", T.StringType(), True),
    T.StructField("Description", T.StringType(), True),
    T.StructField("Quantity", T.IntegerType(), True),
    T.StructField("InvoiceDate", T.StringType(), True),
    T.StructField("UnitPrice", T.DoubleType(), True),
    T.StructField("CustomerID", T.StringType(), True),
    T.StructField("Country", T.StringType(), True),
])


In [0]:
# Lectura do ficheiro incluíndo metadatos
df_raw = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(source_path)
    .select("*", "_metadata")
)

In [0]:
# Conversión mínima de tipos mantendo a idea de capa bronze
df_bronze = (
    df_raw
    .withColumn(
        "InvoiceDate",
        F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm")
    )
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("source_path", F.col("_metadata.file_path"))
    .drop("_metadata")
)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)


## Resultado final

In [0]:
%sql
SELECT *
FROM IDENTIFIER(:target_catalog || '.' || :target_schema || '.' || :target_table)
LIMIT 20;

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,ingestion_timestamp,source_file,source_path
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.65,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.25,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000Z,1.69,13047,United Kingdom,2026-03-17T17:43:31.915Z,online-retail-dataset.csv,dbfs:/Volumes/workspace/default/retail/online-retail-dataset.csv
